# Serial-interface diagnostic condition
Offline-tested only. Start a fresh T4 runtime and upload the updated project ZIP.
Use configs/precommit_hook_qwen_serial.json on BOTH Mac and Colab. Do not launch a full task yet; follow docs/SERIAL_DIAGNOSTICS.md for the prepared requests and exact commands.
This condition changes tool formatting and stopping, pins the observed model revision, and preserves strict validation. It is not Qwen-Agent execution.


# Manual Colab ↔ Mac Pre-commit Hook test

Qwen runs here; task commands run only inside Docker on your Mac. No server, tunnel, remote shell, or background inference worker is started in Colab. Use a GPU runtime. First complete setup below, then start the Mac controller. Transfer one request and one response bundle per turn. This is a nine-call maximum interface evaluation across four cases, not dataset collection. See `docs/SERIAL_DIAGNOSTICS.md`.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, sys, subprocess, json
uploaded = files.upload()  # Upload work/serial_diagnostics_20261029.zip
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    z.extractall('/content/capstone_manual')
ROOT = Path('/content/capstone_manual/ai-safety-capstone')
import hashlib
manifest = json.loads((ROOT / 'PACKAGE_MANIFEST.json').read_text())
for relative, expected in manifest['files_sha256'].items():
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, relative
print('Package source checksums verified:', len(manifest['files_sha256']))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')], check=True)
sys.path.insert(0, str(ROOT / 'src'))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime, timezone
save_dir = Path('/content/drive/MyDrive/ai-safety-capstone') / ('manual_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
save_dir.mkdir(parents=True, exist_ok=False)
from qwen_adapter.local_model import LocalQwen
config = json.loads((ROOT / 'configs/precommit_hook_qwen_serial.json').read_text())
engine = LocalQwen(config['agent']['model'], config['agent']['qwen_options']['generation'])
print(json.dumps(engine.metadata, indent=2))
assert engine.metadata['tool_interface'] == 'qwen_serial_v1'
assert engine.metadata['resolved_revision'] == 'aa8e72537993ba99e69dfaafa59ed015b17504d1'
print('Model loaded. Now start the separate serial diagnostic launcher and export its first request.')


## Repeat these cells for each turn

On your Mac, export the pending request using `scripts/manual_exchange.py export`. Upload that JSON below. The request includes task context/settings; never run commands copied from it in this notebook. Generation can take time; do not click twice. A repeated request in the same Colab session reuses its recorded result instead of regenerating.

In [ ]:
from qwen_adapter.manual_exchange import generate_bundle
uploaded_request = files.upload()  # Upload CALL.request.json exported from your Mac
assert len(uploaded_request) == 1, 'Upload exactly one request'
name, contents = next(iter(uploaded_request.items()))
request = json.loads(contents)
bundle = generate_bundle(request, engine, Path('/content/manual_generation_records'))
call_id = request['call_id']
bundle_path = Path('/content') / f'{call_id}.response_bundle.json'
bundle_path.write_text(json.dumps(bundle, indent=2))
# Persist the complete bundle (request, token records, parsed response/errors) to Drive.
(save_dir / bundle_path.name).write_text(json.dumps(bundle, indent=2))
print('Call:', call_id)
print('Output:', bundle['response'].get('text'))
print('Error:', bundle['response'].get('error'))


In [ ]:
files.download(str(bundle_path))
print('The browser saves this download on your Mac; Colab cannot choose the Mac folder.')
print('After reviewing the response, import it using the correct Mac --run-dir.')
print('The importer archives it in <run-dir>/bundles/' + bundle_path.name)
print('Importing also releases the waiting controller; then export its next request if running.')


The diagnostic launcher stops on the first failure; each case has a fixed two- or three-call cap. A final plain-text report may end it sooner. Preserve all files, including errors. Missing working text is uncertain, not evidence of non-deception. If Colab disconnects after a generation, recover its saved Drive bundle; do not generate the same request again in a fresh runtime without checking whether a result already exists. This notebook does not establish a native CoT channel or execute the task environment.